# Tăng cường đặc trưng VSL

Tạo bản nhiễu từ mẫu gốc trong `data/processed`; `data/raw` giữ nguyên. Đặt `MIN_SAMPLES_PER_CLASS = N` để chỉ bổ sung các lớp có ít hơn N mẫu hoàn chỉnh trong `processed`, tạo đúng số còn thiếu. Đặt `-1` để tăng cường tất cả mẫu gốc theo `COPIES_PER_SAMPLE` mà không giới hạn. Mỗi bản mới ghi `origin_sample_id` để đi cùng fold với mẫu gốc. Validation trong `train.ipynb` chỉ chấm mẫu gốc.


In [17]:
import importlib, sys
from pathlib import Path
import numpy as np
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Chạy Jupyter từ dự án hoặc notebooks/')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
reader = importlib.reload(importlib.import_module('src.data.data_reader'))
augment = importlib.reload(importlib.import_module('src.data.augment'))
DataHandler = reader.DataHandler
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
SELECTED_CLASSES = 'all'  # 'all', 'xin chào', hoặc ['xin chào', 'cảm ơn']
MIN_SAMPLES_PER_CLASS = int(input("MIN_SAMPLES_PER_CLASS (-1 to all): "))  # N; -1 = tăng cường toàn bộ, không giới hạn
COPIES_PER_SAMPLE = 2    # chỉ dùng khi MIN_SAMPLES_PER_CLASS = -1
NOISE_STRENGTH = 1.0    # khoảng (0, 3]
SEED = None  # đặt số nguyên để chạy tái lập
if MIN_SAMPLES_PER_CLASS < -1 or COPIES_PER_SAMPLE < 1 or not 0 < NOISE_STRENGTH <= 3: raise ValueError('Cấu hình ngưỡng/số bản/noise không hợp lệ')
classes = None if SELECTED_CLASSES == 'all' else ([SELECTED_CLASSES] if isinstance(SELECTED_CLASSES, str) else SELECTED_CLASSES)
plan, counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
print(f'{sum(copies for _, copies in plan)} mẫu dự kiến tạo trong {len(counts)} lớp')


720 mẫu dự kiến tạo trong 24 lớp


## Kiểm tra trước khi tạo


In [18]:
for name, count in counts.items():
    extra = sum(copies for source, copies in plan if source.parent.name == name)
    print(f'{name}: hiện có {count}, thêm {extra}' + (f', sau đó có {count + extra}' if MIN_SAMPLES_PER_CLASS != -1 else ''))
print('Đã đủ ngưỡng, không cần tạo thêm.' if not plan else 'Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.')


a: hiện có 100, thêm 30, sau đó có 130
age: hiện có 100, thêm 30, sau đó có 130
b: hiện có 100, thêm 30, sau đó có 130
cảm ơn: hiện có 100, thêm 30, sau đó có 130
dấu huyền: hiện có 100, thêm 30, sau đó có 130
dấu nặng: hiện có 100, thêm 30, sau đó có 130
dấu sắc: hiện có 100, thêm 30, sau đó có 130
dấu â,ô,ê: hiện có 100, thêm 30, sau đó có 130
dấu ơ,ư: hiện có 100, thêm 30, sau đó có 130
g: hiện có 100, thêm 30, sau đó có 130
h: hiện có 100, thêm 30, sau đó có 130
i: hiện có 100, thêm 30, sau đó có 130
k: hiện có 100, thêm 30, sau đó có 130
m: hiện có 100, thêm 30, sau đó có 130
n: hiện có 100, thêm 30, sau đó có 130
o: hiện có 100, thêm 30, sau đó có 130
r: hiện có 100, thêm 30, sau đó có 130
t: hiện có 100, thêm 30, sau đó có 130
ten_la: hiện có 100, thêm 30, sau đó có 130
toi: hiện có 100, thêm 30, sau đó có 130
u: hiện có 100, thêm 30, sau đó có 130
xin chào: hiện có 100, thêm 30, sau đó có 130
y: hiện có 100, thêm 30, sau đó có 130
địa chỉ: hiện có 100, thêm 30, sau đó có 130
Ra

## Tạo mẫu tăng cường
Ô này tính lại số mẫu hiện có ngay trước khi ghi. Với ngưỡng N, chạy lại sau khi đã đủ N sẽ không tạo thêm; với N = -1, mỗi lần chạy tạo thêm bản mới.


In [19]:
rng = np.random.default_rng(SEED)
run_plan, current_counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
created, errors = [], []
for index, (source, copies) in enumerate(run_plan, 1):
    try:
        paths = augment.augment_sample(source, RAW_ROOT, copies, NOISE_STRENGTH, rng)
        created.extend(paths)
        print(f'[{index}/{len(run_plan)}] {source.parent.name}/{source.name} -> {", ".join(path.name for path in paths)}', flush=True)
    except Exception as exc:
        errors.append((str(source), str(exc)))
        print(f'[{index}/{len(run_plan)}] LỖI {source}: {exc}', flush=True)
print(f'Đã tạo {len(created)} mẫu; lỗi {len(errors)}')


[1/321] a/0 -> 100, 101, 102, 103, 104, 105
[2/321] a/1 -> 106, 107, 108, 109, 110, 111
[3/321] a/2 -> 112, 113, 114, 115, 116, 117
[4/321] a/3 -> 118, 119, 120, 121, 122, 123
[5/321] a/4 -> 124, 125, 126, 127, 128, 129
[6/321] age/0 -> 100, 101
[7/321] age/1 -> 102, 103
[8/321] age/10 -> 104, 105
[9/321] age/11 -> 106, 107
[10/321] age/12 -> 108, 109
[11/321] age/13 -> 110, 111
[12/321] age/14 -> 112, 113
[13/321] age/2 -> 114, 115
[14/321] age/3 -> 116, 117
[15/321] age/4 -> 118, 119
[16/321] age/5 -> 120, 121
[17/321] age/6 -> 122, 123
[18/321] age/7 -> 124, 125
[19/321] age/8 -> 126, 127
[20/321] age/9 -> 128, 129
[21/321] b/0 -> 100, 101, 102, 103, 104, 105
[22/321] b/1 -> 106, 107, 108, 109, 110, 111
[23/321] b/2 -> 112, 113, 114, 115, 116, 117
[24/321] b/3 -> 118, 119, 120, 121, 122, 123
[25/321] b/4 -> 124, 125, 126, 127, 128, 129
[26/321] cảm ơn/0 -> 100, 101
[27/321] cảm ơn/1 -> 102, 103
[28/321] cảm ơn/10 -> 104, 105
[29/321] cảm ơn/11 -> 106, 107
[30/321] cảm ơn/12 -> 108, 

## Kiểm tra group


In [20]:
handler = DataHandler()
print('Mẫu đọc được:', handler.validate_dataset())
print('Tổng:', handler.get_class_counts())
print('Group gốc độc lập:', handler.get_origin_counts())
groups = np.asarray(handler.sample_groups('sample'))
augmented = np.asarray([handler.describe(i)['augmented'] for i in range(len(handler))])
assert all((~augmented[groups == group]).any() for group in np.unique(groups))
print(f'{augmented.sum()} bản tăng cường; mọi group đều có mẫu gốc')
print('Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.')


Mẫu đọc được: 3120
Tổng: {'a': 130, 'age': 130, 'b': 130, 'cảm ơn': 130, 'dấu huyền': 130, 'dấu nặng': 130, 'dấu sắc': 130, 'dấu â,ô,ê': 130, 'dấu ơ,ư': 130, 'g': 130, 'h': 130, 'i': 130, 'k': 130, 'm': 130, 'n': 130, 'o': 130, 'r': 130, 't': 130, 'ten_la': 130, 'toi': 130, 'u': 130, 'xin chào': 130, 'y': 130, 'địa chỉ': 130}
Group gốc độc lập: {'a': 5, 'age': 15, 'b': 5, 'cảm ơn': 15, 'dấu huyền': 5, 'dấu nặng': 15, 'dấu sắc': 5, 'dấu â,ô,ê': 5, 'dấu ơ,ư': 5, 'g': 15, 'h': 30, 'i': 15, 'k': 15, 'm': 30, 'n': 15, 'o': 5, 'r': 15, 't': 15, 'ten_la': 15, 'toi': 15, 'u': 15, 'xin chào': 15, 'y': 16, 'địa chỉ': 15}
2799 bản tăng cường; mọi group đều có mẫu gốc
Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.
